# HUẤN LUYỆN (FINE-TUNE) YOLOV8 NHẬN DIỆN MŨ BẢO HIỂM TRÊN GOOGLE COLAB
**Dự án:** Traffic Violation Detection System (TVDS)
**Dataset:** `helmet-detection-vn` (92 ảnh thực tế đã gán nhãn)

**LƯU Ý QUAN TRỌNG:**
Trước khi chạy, hãy kiểm tra menu: **Runtime -> Change runtime type -> Chọn T4 GPU** rồi bấm **Save**.

In [ ]:
# 1. KIỂM TRA GPU
!nvidia-smi

In [ ]:
# 2. CÀI ĐẶT CÁC THƯ VIỆN CẦN THIẾT
!pip install -q ultralytics roboflow

In [ ]:
# 3. TẢI DATASET TRỰC TIẾP TỪ ROBOFLOW
from roboflow import Roboflow

rf = Roboflow(api_key="uChxghdGBdHx3AVEfJA8")
project = rf.workspace("phuc-sgu-dang-hoang").project("helmet-detection-vn")
version = project.version(1)
dataset = version.download("yolov8")

print("Đã tải dataset thành công tại:", dataset.location)

In [ ]:
# 4. BẮT ĐẦU HUẤN LUYỆN (FINE-TUNE)
import os
from ultralytics import YOLO

# Nếu bạn đã upload file helmet_best.pt lên Colab thì nó sẽ fine-tune tiếp từ model cũ.
# Nếu chưa upload thì nó sẽ dùng yolov8n.pt chuẩn.
weights_source = "helmet_best.pt" if os.path.exists("helmet_best.pt") else "yolov8n.pt"
print(f"[*] Đang khởi tạo mô hình từ: {weights_source}")

model = YOLO(weights_source)

# Train 50 epochs (imgsz=640, lr0=0.0015 theo chuẩn fine-tune từ helmet_best.pt;
# dataset phải giữ nguyên thứ tự 7 class 0..6 kể cả typo 'helemt' và trộn với 368 ảnh train gốc)
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    patience=15,
    optimizer="auto",
    lr0=0.0015,
    plots=True
)

In [ ]:
# 5. ĐÁNH GIÁ CHỈ SỐ ĐỘ CHÍNH XÁC (mAP)
metrics = model.val()
print("[*] Độ chính xác mAP@50:", metrics.box.map50)
print("[*] Độ chính xác mAP@50-95:", metrics.box.map)

In [ ]:
# 6. TẢI FILE WEIGHTS ĐÃ TRAIN TỐT NHẤT VỀ MÁY
from google.colab import files
best_weights_path = "runs/detect/train/weights/best.pt"
if os.path.exists(best_weights_path):
    print("[*] Đang tải best.pt về máy tính của bạn...")
    files.download(best_weights_path)
else:
    print("Chưa tìm thấy best.pt, hãy kiểm tra thư mục runs/detect/")